# UAV dense-canopy navigation: height-aware fine-tuning + evaluation on Colab

The lab machine is down, so this notebook tests the novelty by **fine-tuning the authors' released
DiffPhysDrone policy** (already in `MyDrive/uav_eval/checkpoints/released_base.pth`) in two ways,
then evaluating all three on the height-change test and the standard scenes:

* `ft_fix`: fine-tuned with the ground-affinity loss fix only (control)
* `ft_height`: fine-tuned with the loss fix **and** height-aware targets (the novelty)

Same scenes and settings as the lab runs. Trained models and every finished flight are saved to Google
Drive, so if Colab disconnects you just run all cells again and it continues where it stopped.

**Steps:** `Runtime` > `Change runtime type` > **T4 GPU** > Save, then `Runtime` > **Run all**
(allow Drive access).

Results land in `MyDrive/uav_eval/benchmark_results/` and the final tables in `MyDrive/uav_eval/summary/`.

In [ ]:
# 1. GPU check
import subprocess
r = subprocess.run("nvidia-smi --query-gpu=name,memory.total,temperature.gpu,utilization.gpu --format=csv",
                   shell=True, capture_output=True, text=True)
if r.returncode != 0:
    raise SystemExit("NO GPU: Runtime > Change runtime type > T4 GPU > Save, then Runtime > Run all")
print(r.stdout)

In [ ]:
# 2. Google Drive, folders, checkpoints
import hashlib, os, shutil, subprocess, sys, time
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive/uav_eval")
RES = DRIVE / "benchmark_results"          # every flight is written here (survives disconnects)
SUMMARY = DRIVE / "summary"
CKPT_DRIVE = DRIVE / "checkpoints"
CKPT = Path("/content/checkpoints")         # local copy, faster to load
REPO = Path("/content/uav")
for p in (RES, SUMMARY, CKPT_DRIVE, CKPT):
    p.mkdir(parents=True, exist_ok=True)

KNOWN_SHA256 = {   # from benchmark_docs/lab_results/checkpoint_sha256.txt (lab machine)
    "ours_20k.pth": "00b402a097d0d01c108dce0a23fb1155cbea751d254ee434efe4f31dac268731",
    "ours_50k.pth": "8290ef52789abaefd7fb4a4c206d4692d95c4c7009f365c63564097494b0aa7e",
    "released_base.pth": "81418565033e09a7c2c8242c585c8e8cb4413051fb1a2fecce09499458afa4d3",
}
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

HAVE = set()
for name in ["released_base.pth", "ft_height.pth", "ft_fix.pth"]:
    src = CKPT_DRIVE / name
    if not src.exists():
        print(f"missing  {name}")
        continue
    shutil.copy2(src, CKPT / name)
    digest = sha256(CKPT / name)
    note = ("" if name not in KNOWN_SHA256 else
            "  (matches lab)" if digest == KNOWN_SHA256[name] else "  WARNING: differs from the lab file")
    print(f"found    {name}  sha256 {digest[:16]}...{note}")
    HAVE.add(name[:-4])
if "released_base" not in HAVE:
    raise SystemExit("released_base.pth is missing from MyDrive/uav_eval/checkpoints")

In [ ]:
TRAIN_PATCH_TEXT = 'diff --git a/env_cuda.py b/env_cuda.py\nindex a894de3..7815695 100644\n--- a/env_cuda.py\n+++ b/env_cuda.py\n@@ -43,7 +43,7 @@ run = RunFunction.apply\n class Env:\n     def __init__(self, batch_size, width, height, grad_decay, device=\'cpu\', fov_x_half_tan=0.53,\n                  single=False, gate=False, ground_voxels=False, scaffold=False, speed_mtp=1,\n-                 random_rotation=False, cam_angle=10) -> None:\n+                 random_rotation=False, cam_angle=10, target_dz=0.0) -> None:\n         self.device = device\n         self.batch_size = batch_size\n         self.width = width\n@@ -87,6 +87,7 @@ class Env:\n         ], device=device).repeat(batch_size // 8 + 7, 1)[:batch_size]\n         self.flow = torch.empty((batch_size, 0, height, width), device=device)\n         self.single = single\n+        self.target_dz = target_dz\n         self.gate = gate\n         self.ground_voxels = ground_voxels\n         self.scaffold = scaffold\n@@ -196,6 +197,10 @@ class Env:\n             torch.rand_like(scale) - 0.5], -1)\n         self.p = self.p_init * scale + torch.randn_like(scale) * 0.1\n         self.p_target = self.p_end * scale + torch.randn_like(scale) * 0.1\n+        if self.target_dz > 0:\n+            dz = (torch.rand((B, 1), device=device) * 2 - 1) * self.target_dz\n+            self.p_target = torch.cat([self.p_target[:, :2],\n+                                       (self.p_target[:, 2:] + dz).clamp(-0.6, 1.5)], -1)\n \n         if self.random_rotation:\n             yaw_bias = torch.rand(B//self.n_drones_per_group, device=device).repeat_interleave(self.n_drones_per_group, 0) * 1.5 - 0.75\ndiff --git a/main_cuda.py b/main_cuda.py\nindex 2911b85..a0cd02e 100644\n--- a/main_cuda.py\n+++ b/main_cuda.py\n@@ -34,6 +34,12 @@ parser.add_argument(\'--speed_mtp\', type=float, default=1.0)\n parser.add_argument(\'--fov_x_half_tan\', type=float, default=0.53)\n parser.add_argument(\'--timesteps\', type=int, default=150)\n parser.add_argument(\'--cam_angle\', type=int, default=10)\n+parser.add_argument(\'--ground_affinity_fix\', action=\'store_true\',\n+                    help=\'multiply (not add) coef_ground_affinity and the ground-affinity loss\')\n+parser.add_argument(\'--coef_height\', type=float, default=0.0,\n+                    help=\'weight of the ramped height-tracking loss (0 = off)\')\n+parser.add_argument(\'--target_dz\', type=float, default=0.0,\n+                    help=\'random target height offset U(-dz, dz) in metres (0 = upstream)\')\n parser.add_argument(\'--single\', default=False, action=\'store_true\')\n parser.add_argument(\'--gate\', default=False, action=\'store_true\')\n parser.add_argument(\'--ground_voxels\', default=False, action=\'store_true\')\n@@ -47,7 +53,7 @@ print(args)\n \n device = torch.device(\'cuda\')\n \n-env = Env(args.batch_size, 64, 48, args.grad_decay, device,\n+env = Env(args.batch_size, 64, 48, args.grad_decay, device, target_dz=args.target_dz,\n           fov_x_half_tan=args.fov_x_half_tan, single=args.single,\n           gate=args.gate, ground_voxels=args.ground_voxels,\n           scaffold=args.scaffold, speed_mtp=args.speed_mtp,\n@@ -166,6 +172,9 @@ for i in pbar:\n \n     p_history = torch.stack(p_history)\n     loss_ground_affinity = p_history[..., 2].relu().pow(2).mean()\n+    height_ramp = torch.linspace(0, 1, p_history.shape[0], device=p_history.device)[:, None]\n+    loss_height = (height_ramp * F.smooth_l1_loss(\n+        p_history[..., 2], env.p_target[None, :, 2].expand_as(p_history[..., 2]), reduction=\'none\')).mean()\n     act_buffer = torch.stack(act_buffer)\n \n     v_history = torch.stack(v_history)\n@@ -210,7 +219,9 @@ for i in pbar:\n         args.coef_speed * loss_speed + \\\n         args.coef_v_pred * loss_v_pred + \\\n         args.coef_collide * loss_collide + \\\n-        args.coef_ground_affinity + loss_ground_affinity\n+        (args.coef_ground_affinity * loss_ground_affinity if args.ground_affinity_fix\n+         else args.coef_ground_affinity + loss_ground_affinity) + \\\n+        args.coef_height * loss_height\n \n     if torch.isnan(loss):\n         print("loss is nan, exiting...")\n@@ -239,6 +250,7 @@ for i in pbar:\n             \'loss_speed\': loss_speed,\n             \'loss_collide\': loss_collide,\n             \'loss_ground_affinity\': loss_ground_affinity,\n+            \'loss_height\': loss_height,\n             \'success\': _success,\n             \'max_speed\': speed_history.max(0).values.mean(),\n             \'avg_speed\': avg_speed.mean(),\n'

# 3. Code: team repo at the merged PR #3 commit + DiffPhysDrone at the pinned commit + PyTorch fix
def sh(cmd, cwd=None, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, cwd=cwd, text=True, capture_output=True)
    out = (r.stdout + r.stderr).strip()
    if out:
        print(out[-3000:])
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed ({r.returncode}): {cmd}")
    return r

TEAM_COMMIT = "91d186a"                                   # main after PR #3 (keeps resume fingerprints stable)
SIM_COMMIT = "271936190b5c2a5e760230e718fa6b167718a5bf"   # DiffPhysDrone revision pinned by the benchmark
if not (REPO / ".git").exists():
    sh(f"git clone -q https://github.com/Pooja-Satpathy09/uav-dense-canopy-navigation.git {REPO}")
sh(f"git checkout -q {TEAM_COMMIT}", cwd=REPO)
sh("git submodule update --init simulator/DiffPhysDrone", cwd=REPO)
SIM = REPO / "simulator" / "DiffPhysDrone"
assert sh("git rev-parse HEAD", cwd=SIM).stdout.strip() == SIM_COMMIT
PATCH = REPO / "benchmark_docs/patches/diffphysdrone_pytorch27_scalar_type.patch"
quiet = lambda cmd: subprocess.run(cmd, shell=True, cwd=SIM, capture_output=True).returncode == 0
if quiet(f"git apply --check {PATCH}"):
    sh(f"git apply {PATCH}", cwd=SIM)
    print("PyTorch compatibility patch applied")
elif quiet(f"git apply --reverse --check {PATCH}"):
    print("PyTorch compatibility patch already applied")
else:
    raise RuntimeError("patch neither applies nor is already applied")
TRAIN_PATCH = Path("/content/diffphysdrone_height_aware.patch")
TRAIN_PATCH.write_text(TRAIN_PATCH_TEXT)
if quiet(f"git apply --check {TRAIN_PATCH}"):
    sh(f"git apply {TRAIN_PATCH}", cwd=SIM)
    main = SIM / "main_cuda.py"      # save the final model too (upstream only saves every 10k iterations)
    main.write_text(main.read_text() + "\ntorch.save(model.state_dict(), 'checkpoint_final.pth')\n")
    print("height-aware training patch applied")
elif "ground_affinity_fix" in (SIM / "main_cuda.py").read_text():
    print("height-aware training patch already applied")
else:
    raise RuntimeError("training patch does not apply")

In [ ]:
# 4. Build the DiffPhysDrone CUDA simulator (about 3-6 minutes the first time in a session)
import torch
def quadsim_ok():
    return subprocess.run([sys.executable, "-c", "import torch, quadsim_cuda"], capture_output=True).returncode == 0

if quadsim_ok():
    print("quadsim_cuda already built")
else:
    cap = torch.cuda.get_device_capability()
    arch = f"{cap[0]}.{cap[1]}"
    nvcc = subprocess.run("nvcc --version || /usr/local/cuda/bin/nvcc --version", shell=True,
                          capture_output=True, text=True).stdout
    print(f"torch {torch.__version__} (CUDA {torch.version.cuda}), GPU arch {arch}")
    print(nvcc.strip().splitlines()[-1] if nvcc else "nvcc not found")
    sh(f"{sys.executable} -m pip -q install ninja", check=False)
    os.environ["TORCH_CUDA_ARCH_LIST"] = arch
    os.environ.setdefault("CUDA_HOME", "/usr/local/cuda")
    os.environ["PATH"] = "/usr/local/cuda/bin:" + os.environ["PATH"]
    r = sh(f"{sys.executable} -m pip install --no-build-isolation ./src", cwd=SIM, check=False)
    if r.returncode != 0 or not quadsim_ok():
        raise RuntimeError("Build failed. Copy the last 30 lines of the output above and send them to Claude.")
    print("quadsim_cuda built OK")

In [ ]:
# 5. Test scenes: regenerate the lab scenes and check they are byte-identical, then the height scenes
import json
tools = REPO / "lab_tools"
EMBEDDED = {
"ten_metrics.py": "\"\"\"Ten-metric statistics for one or more `experiment` result folders.\n\nFor every result folder (one checkpoint under one plan) this prints each benchmark\nmetric with its 95% confidence interval, and for every requested pair of folders a\npaired test on the same flights (same scene and speed):\n\n  binary metrics (goal_success, collision)  exact McNemar test\n  continuous metrics                        Wilcoxon signed-rank test\n  multiple comparisons                      Holm correction over the 8 flight metrics\n\nConfidence intervals: Wilson for rates, scene-cluster bootstrap (2000 resamples) for means.\nThe two scene metrics (traversability, relative_gap_size) describe the scenes, not the\npolicy, so they are reported once per difficulty.\n\nUsage:\n  python ten_metrics.py RESULTS_DIR NAME [NAME ...] [--pair A B ...] [--out PREFIX]\nExample:\n  python lab_tools/ten_metrics.py benchmark_results fix_20k_height height_20k_height \\\n      --pair fix_20k_height height_20k_height --out benchmark_results/ten_metrics_height\n\"\"\"\nimport argparse\nimport csv\nimport json\nimport math\nimport random\nimport statistics\nfrom collections import defaultdict\nfrom pathlib import Path\n\nFLIGHT = [(\"goal_success\", \"rate\", \"higher\"), (\"collision\", \"rate\", \"lower\"),\n          (\"min_obstacle_clearance_m\", \"m\", \"higher\"), (\"path_excess_factor\", \"1\", \"lower\"),\n          (\"average_speed_mps\", \"m/s\", \"\"), (\"relative_end_distance\", \"1\", \"lower\"),\n          (\"speed_rmse_mps\", \"m/s\", \"lower\"), (\"jerk_rms_mps3\", \"m/s^3\", \"lower\")]\nSCENE = [(\"traversability\", \"1\"), (\"relative_gap_size\", \"1\")]\nBINARY = {\"goal_success\", \"collision\"}\n\n\ndef wilson(k, n, z=1.959963984540054):\n    if n == 0:\n        return float(\"nan\"), float(\"nan\")\n    p, den = k / n, 1 + z * z / n\n    c, h = (p + z * z / (2 * n)) / den, z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den\n    return max(0.0, c - h), min(1.0, c + h)\n\n\ndef bootstrap_mean_ci(pairs, resamples=2000):\n    \"\"\"pairs: [(scene_id, value)]; resample whole scenes (both speeds stay together).\"\"\"\n    clusters = defaultdict(list)\n    for sid, v in pairs:\n        clusters[sid].append(v)\n    ids, rng, means = sorted(clusters), random.Random(2026), []\n    if len(ids) < 2:\n        return float(\"nan\"), float(\"nan\")\n    for _ in range(resamples):\n        picked = [v for _ in ids for v in clusters[rng.choice(ids)]]\n        means.append(statistics.fmean(picked))\n    means.sort()\n    return means[int(0.025 * (resamples - 1))], means[int(0.975 * (resamples - 1))]\n\n\ndef mcnemar_exact(b, c):\n    n = b + c\n    if n == 0:\n        return 1.0\n    return min(1.0, 2 * sum(math.comb(n, i) for i in range(min(b, c) + 1)) / 2 ** n)\n\n\ndef wilcoxon_p(diffs):\n    diffs = [d for d in diffs if d != 0]\n    if not diffs:\n        return 1.0\n    try:\n        from scipy.stats import wilcoxon\n        return float(wilcoxon(diffs).pvalue)\n    except Exception:                                    # normal approximation fallback\n        n = len(diffs)\n        order = sorted(range(n), key=lambda i: abs(diffs[i]))\n        ranks, i = [0.0] * n, 0\n        while i < n:\n            j = i\n            while j + 1 < n and abs(diffs[order[j + 1]]) == abs(diffs[order[i]]):\n                j += 1\n            for k in range(i, j + 1):\n                ranks[order[k]] = (i + j) / 2 + 1\n            i = j + 1\n        w = sum(r for r, d in zip(ranks, diffs) if d > 0)\n        mu, sd = n * (n + 1) / 4, math.sqrt(n * (n + 1) * (2 * n + 1) / 24)\n        z = (abs(w - mu) - 0.5) / sd if sd else 0.0\n        return math.erfc(z / math.sqrt(2))\n\n\ndef holm(pvalues):\n    m, order = len(pvalues), sorted(range(len(pvalues)), key=lambda i: pvalues[i])\n    adjusted, running = [1.0] * m, 0.0\n    for rank, i in enumerate(order):\n        running = max(running, min(1.0, (m - rank) * pvalues[i]))\n        adjusted[i] = running\n    return adjusted\n\n\ndef load(results_dir, name):\n    flights = {}\n    for path in sorted((Path(results_dir) / name).glob(\"run_[0-9][0-9][0-9][0-9][0-9]/result.metrics.json\")):\n        r = json.loads(path.read_text())\n        key = (r[\"scene_id\"], float(r[\"config\"][\"target_speed_mps\"]))\n        flights[key] = {\"difficulty\": r[\"difficulty\"], \"termination\": r[\"termination\"],\n                        **{m: r[\"metrics\"].get(m, {}).get(\"value\") for m, *_ in FLIGHT + SCENE}}\n    if not flights:\n        raise SystemExit(f\"No result.metrics.json found under {Path(results_dir) / name}\")\n    return flights\n\n\ndef fmt(x, digits=3):\n    return \"n/a\" if x is None or (isinstance(x, float) and math.isnan(x)) else f\"{x:.{digits}f}\"\n\n\ndef per_run_rows(name, flights):\n    rows = []\n    for metric, unit, better in FLIGHT:\n        pairs = [(k[0], float(f[metric])) for k, f in flights.items() if f[metric] is not None]\n        n = len(pairs)\n        if metric in BINARY:\n            k = sum(v for _, v in pairs)\n            lo, hi = wilson(int(k), n)\n            rows.append({\"run\": name, \"metric\": metric, \"unit\": unit, \"better\": better, \"n\": n,\n                         \"estimate\": k / n if n else float(\"nan\"), \"sd\": \"\", \"median\": \"\",\n                         \"ci_low\": lo, \"ci_high\": hi, \"ci_method\": \"Wilson\"})\n        else:\n            vals = [v for _, v in pairs]\n            lo, hi = bootstrap_mean_ci(pairs)\n            rows.append({\"run\": name, \"metric\": metric, \"unit\": unit, \"better\": better, \"n\": n,\n                         \"estimate\": statistics.fmean(vals) if vals else float(\"nan\"),\n                         \"sd\": statistics.stdev(vals) if n > 1 else float(\"nan\"),\n                         \"median\": statistics.median(vals) if vals else float(\"nan\"),\n                         \"ci_low\": lo, \"ci_high\": hi, \"ci_method\": \"scene bootstrap\"})\n    return rows\n\n\ndef pair_rows(a_name, a, b_name, b):\n    keys = sorted(set(a) & set(b))\n    rows = []\n    for metric, unit, better in FLIGHT:\n        both = [(a[k][metric], b[k][metric]) for k in keys\n                if a[k][metric] is not None and b[k][metric] is not None]\n        n = len(both)\n        if metric in BINARY:\n            only_a = sum(1 for x, y in both if x and not y)\n            only_b = sum(1 for x, y in both if y and not x)\n            p, test = mcnemar_exact(only_a, only_b), f\"McNemar (A only {only_a}, B only {only_b})\"\n        else:\n            p, test = wilcoxon_p([float(y) - float(x) for x, y in both]), \"Wilcoxon signed-rank\"\n        rows.append({\"A\": a_name, \"B\": b_name, \"metric\": metric, \"unit\": unit, \"better\": better,\n                     \"n_pairs\": n, \"mean_A\": statistics.fmean(float(x) for x, _ in both) if n else float(\"nan\"),\n                     \"mean_B\": statistics.fmean(float(y) for _, y in both) if n else float(\"nan\"),\n                     \"test\": test, \"p\": p})\n    for row, adj in zip(rows, holm([r[\"p\"] for r in rows])):\n        row[\"p_holm\"] = adj\n    return rows\n\n\ndef main():\n    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    ap.add_argument(\"results_dir\")\n    ap.add_argument(\"names\", nargs=\"+\")\n    ap.add_argument(\"--pair\", nargs=2, action=\"append\", default=[], metavar=(\"A\", \"B\"))\n    ap.add_argument(\"--out\", help=\"write PREFIX.md, PREFIX_per_run.csv and PREFIX_paired.csv\")\n    args = ap.parse_args()\n    data = {n: load(args.results_dir, n) for n in dict.fromkeys(args.names + [x for p in args.pair for x in p])}\n\n    lines = [\"## Ten-metric statistics\", \"\",\n             \"Rates: Wilson 95% CI. Means: 95% CI from a scene-cluster bootstrap (2000 resamples).\", \"\"]\n    per_run = []\n    for name, flights in data.items():\n        rows = per_run_rows(name, flights)\n        per_run += rows\n        lines += [f\"### {name} ({len(flights)} flights)\", \"\",\n                  \"| Metric | Unit | Better | Estimate | 95% CI | SD | Median |\", \"|---|---|---|---|---|---|---|\"]\n        for r in rows:\n            lines.append(f\"| {r['metric']} | {r['unit']} | {r['better'] or '-'} | {fmt(r['estimate'])} | \"\n                         f\"[{fmt(r['ci_low'])}, {fmt(r['ci_high'])}] | {fmt(r['sd']) if r['sd'] != '' else '-'} | \"\n                         f\"{fmt(r['median']) if r['median'] != '' else '-'} |\")\n        lines.append(\"\")\n\n    first = next(iter(data.values()))\n    scenes = {}\n    for (sid, _), f in first.items():\n        scenes[sid] = f\n    lines += [\"### Scene metrics (properties of the test scenes, same for every checkpoint)\", \"\",\n              \"| Difficulty | Scenes | traversability (mean) | relative_gap_size (mean) |\", \"|---|---|---|---|\"]\n    by_diff = defaultdict(list)\n    for f in scenes.values():\n        by_diff[f[\"difficulty\"]].append(f)\n    for diff in sorted(by_diff):\n        g = by_diff[diff]\n        cells = []\n        for m, _ in SCENE:\n            vals = [float(x[m]) for x in g if x[m] is not None]\n            cells.append(fmt(statistics.fmean(vals)) if vals else \"n/a\")\n        lines.append(f\"| {diff} | {len(g)} | {cells[0]} | {cells[1]} |\")\n    lines.append(\"\")\n\n    paired = []\n    for a_name, b_name in args.pair:\n        rows = pair_rows(a_name, data[a_name], b_name, data[b_name])\n        paired += rows\n        lines += [f\"### Paired: A = {a_name} vs B = {b_name} (same scene and speed)\", \"\",\n                  \"| Metric | Better | Mean A | Mean B | n | Test | p | p (Holm) |\", \"|---|---|---|---|---|---|---|---|\"]\n        for r in rows:\n            star = \" *\" if r[\"p_holm\"] < 0.05 else \"\"\n            lines.append(f\"| {r['metric']} | {r['better'] or '-'} | {fmt(r['mean_A'])} | {fmt(r['mean_B'])} | \"\n                         f\"{r['n_pairs']} | {r['test']} | {fmt(r['p'], 4)} | {fmt(r['p_holm'], 4)}{star} |\")\n        lines += [\"\", \"* = significant at 0.05 after Holm correction over the 8 flight metrics.\", \"\"]\n\n    text = \"\\n\".join(lines)\n    print(text)\n    if args.out:\n        out = Path(args.out)\n        out.parent.mkdir(parents=True, exist_ok=True)\n        out.with_suffix(\".md\").write_text(text + \"\\n\")\n        for suffix, rows in ((\"_per_run.csv\", per_run), (\"_paired.csv\", paired)):\n            if rows:\n                with open(f\"{out}{suffix}\", \"w\", newline=\"\") as fh:\n                    w = csv.DictWriter(fh, fieldnames=list(rows[0]))\n                    w.writeheader()\n                    w.writerows(rows)\n        print(f\"\\nWrote {out.with_suffix('.md')} and CSV files next to it.\")\n\n\nif __name__ == \"__main__\":\n    main()\n",
"height_table.py": "\"\"\"Height-change test summary (novelty evaluation).\n\nIn benchmark_scenes_height the goal is 0.6 m above the start for even scene seeds and\n0.6 m below it for odd seeds. Reads benchmark_results/<name>_goal_plane.csv (made by\nrescore_goal_plane.py) and prints, per checkpoint and for goal-up / goal-down flights:\ngoal-sphere success (Wilson 95% CI), safe success, collisions and the height error where\nthe drone reached the goal plane. Then paired exact McNemar tests on goal-sphere success.\n\nUsage: python lab_tools/height_table.py NAME [NAME ...] [--pair A B ...]\nRun from the folder that contains benchmark_results/.\n\"\"\"\nimport argparse\nimport csv\nimport math\nimport re\nimport statistics\n\n\ndef wilson(k, n, z=1.96):\n    if n == 0:\n        return float(\"nan\"), float(\"nan\")\n    p, d = k / n, 1 + z * z / n\n    c, h = (p + z * z / (2 * n)) / d, z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d\n    return max(0.0, c - h), min(1.0, c + h)\n\n\ndef mcnemar(b, c):\n    n = b + c\n    return 1.0 if n == 0 else min(1.0, 2 * sum(math.comb(n, i) for i in range(min(b, c) + 1)) / 2 ** n)\n\n\ndef load(name):\n    rows = list(csv.DictReader(open(f\"benchmark_results/{name}_goal_plane.csv\")))\n    for r in rows:\n        r[\"direction\"] = \"up\" if int(re.search(r\"_seed_(\\d+)\", r[\"scene_id\"]).group(1)) % 2 == 0 else \"down\"\n    return rows\n\n\ndef ok(r):\n    return r[\"goal_success\"] == \"1\" or r[\"plane_success\"] == \"1\"\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"names\", nargs=\"+\")\n    ap.add_argument(\"--pair\", nargs=2, action=\"append\", default=[])\n    args = ap.parse_args()\n    data = {}\n    for name in dict.fromkeys(args.names + [x for p in args.pair for x in p]):\n        try:\n            data[name] = load(name)\n        except FileNotFoundError:\n            print(f\"{name}: no results yet (run rescore_goal_plane.py first)\")\n    print(\"Goal 0.6 m above (up) or below (down) the start. Height error = z - goal z at the goal plane.\")\n    print(f\"{'checkpoint':<22} {'goal':<5} {'n':>4} | {'goal-sphere success':<22} {'safe':>5} {'coll':>5} | \"\n          f\"{'median height err':>17} {'median |err|':>12}\")\n    for name, rows in data.items():\n        for direction in (\"up\", \"down\", \"all\"):\n            g = rows if direction == \"all\" else [r for r in rows if r[\"direction\"] == direction]\n            n = len(g)\n            if not n:\n                continue\n            k = sum(r[\"goal_success\"] == \"1\" for r in g)\n            lo, hi = wilson(k, n)\n            err = [float(r[\"miss_vertical_m\"]) for r in g if r[\"plane_success\"] == \"1\" and r[\"miss_vertical_m\"] != \"\"]\n            med = f\"{statistics.median(err):+.2f} m\" if err else \"n/a\"\n            mabs = f\"{statistics.median(abs(e) for e in err):.2f} m\" if err else \"n/a\"\n            coll = sum(r[\"termination\"] == \"collision\" for r in g)\n            print(f\"{name:<22} {direction:<5} {n:>4} | {k / n:.2f} [{lo:.2f}, {hi:.2f}]{'':6} \"\n                  f\"{sum(ok(r) for r in g) / n:>5.2f} {coll:>5} | {med:>17} {mabs:>12}\")\n        print()\n    for a_name, b_name in args.pair:\n        if a_name not in data or b_name not in data:\n            continue\n        A = {(r[\"scene_id\"], r[\"speed_mps\"]): r for r in data[a_name]}\n        B = {(r[\"scene_id\"], r[\"speed_mps\"]): r for r in data[b_name]}\n        keys = sorted(set(A) & set(B))\n        print(f\"Paired goal-sphere success: A = {a_name}, B = {b_name}\")\n        for direction in (\"up\", \"down\", \"all\"):\n            ks = [k for k in keys if direction == \"all\" or A[k][\"direction\"] == direction]\n            ga = sum(A[k][\"goal_success\"] == \"1\" for k in ks)\n            gb = sum(B[k][\"goal_success\"] == \"1\" for k in ks)\n            only_a = sum(A[k][\"goal_success\"] == \"1\" and B[k][\"goal_success\"] != \"1\" for k in ks)\n            only_b = sum(B[k][\"goal_success\"] == \"1\" and A[k][\"goal_success\"] != \"1\" for k in ks)\n            n = max(1, len(ks))\n            print(f\"  {direction:<5} n={len(ks):>3}  A {ga / n:.2f}  B {gb / n:.2f}  \"\n                  f\"(A only {only_a}, B only {only_b})  exact McNemar p = {mcnemar(only_a, only_b):.4f}\")\n        print()\n\n\nif __name__ == \"__main__\":\n    main()\n",
"make_height_scenes.py": "import json, re, sys\nfrom pathlib import Path\nsrc, dst, dz = Path(sys.argv[1]), Path(sys.argv[2]), float(sys.argv[3])\ndst.mkdir(parents=True, exist_ok=True)\nfiles = sorted(src.glob(\"*_seed_*.json\"))\nfor f in files:\n    s = json.loads(f.read_text())\n    seed = int(re.search(r\"_seed_(\\d+)\", f.name).group(1))\n    sign = 1.0 if seed % 2 == 0 else -1.0\n    s[\"goal\"][2] = s[\"start\"][2] + sign * dz\n    s[\"goal_height_change_m\"] = sign * dz\n    (dst / f.name).write_text(json.dumps(s, indent=2))\nprint(f\"Wrote {len(files)} scenes to {dst}: even seeds goal +{dz} m, odd seeds goal -{dz} m\")\n"
}
for name, text in EMBEDDED.items():
    (tools / name).write_text(text)

sh("python -m uav_benchmark generate-scenes --repo . --out benchmark_scenes --start-seed 0 --count 30", cwd=REPO)
sh("python lab_tools/shift_scenes.py benchmark_scenes benchmark_scenes_ground -1.0", cwd=REPO)
lab = json.loads((REPO / "benchmark_docs/lab_results/ours_50k_fov82_experiment.json").read_text())["scenes"]
same = sum(sha256(REPO / rel) == h for rel, h in lab.items())
print(f"{same}/{len(lab)} scenes identical to the lab scenes")
assert same == len(lab) == 90, "scenes differ from the lab: stop and tell Claude"

sh("python lab_tools/make_height_scenes.py benchmark_scenes_ground benchmark_scenes_height 0.6", cwd=REPO)
h = hashlib.sha256()
for f in sorted((REPO / "benchmark_scenes_height").glob("*.json")):
    h.update(f.name.encode()); h.update(f.read_bytes())
assert h.hexdigest() == "932bab0cbb5ed69b68669eb75697c5e5b38dbbcefc89380d1e8f0a157be6d814", "height scenes differ"
print("90 height scenes OK (goal 0.6 m above the start for even seeds, 0.6 m below for odd seeds)")
sh("python -m uav_benchmark validate-scenes --repo . --inputs benchmark_scenes_height "
   "--config benchmark_configs/eval_24s.json --out /content/scene_validation_height.json", cwd=REPO)

plan = json.loads((REPO / "benchmark_configs/lab_plan_fov82.json").read_text())
(REPO / "benchmark_configs/lab_plan_height_fov82.json").write_text(
    json.dumps({**plan, "scene_glob": "benchmark_scenes_height/*_seed_*.json"}, indent=2))
(REPO / "benchmark_configs/check_hard10_fov82.json").write_text(
    json.dumps({**plan, "scene_glob": "benchmark_scenes_ground/hard_seed_?.json"}, indent=2))
for name in ("ours_20k_fov82", "ours_50k_fov82", "released_base_fov82"):   # lab results, for comparison
    shutil.copy2(REPO / f"benchmark_docs/lab_results/{name}_goal_plane.csv", RES / f"lab_{name}_goal_plane.csv")
print("plans written; lab result tables copied as lab_*_goal_plane.csv")

In [ ]:
# 6. Runner (resumable). Finished flights are skipped, so re-running is always safe.
import re
def gpu_temp():
    out = subprocess.run(["nvidia-smi", "--query-gpu=temperature.gpu", "--format=csv,noheader,nounits"],
                         capture_output=True, text=True).stdout.strip()
    return int(out.splitlines()[0]) if out else None

def evaluate(ckpt, plan, out):
    t = gpu_temp()
    cmd = [sys.executable, "-u", "-m", "uav_benchmark", "experiment", "--repo", str(REPO),
           "--checkpoint", str(CKPT / f"{ckpt}.pth"), "--plan", f"benchmark_configs/{plan}.json",
           "--out", str(RES / out), "--resume"]
    start = time.time()
    print(f"=== {out}: {ckpt} on {plan} (GPU {t} C)")
    p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    tail = []
    for line in p.stdout:
        tail = (tail + [line])[-20:]
        m = re.match(r"(\d+)/(\d+) ", line)
        if "ERROR" in line or (m and (int(m.group(1)) % 30 == 0 or m.group(1) == m.group(2))):
            print(f"  [{(time.time() - start) / 60:5.1f} min] {line.rstrip()}", flush=True)
    if p.wait() != 0:
        print("".join(tail))
        raise RuntimeError(f"{out} failed; send the lines above to Claude")
    done = len(list((RES / out).glob("run_[0-9][0-9][0-9][0-9][0-9]/result.metrics.json")))
    print(f"=== {out}: {done} flights saved, {(time.time() - start) / 60:.1f} min this session\n")
    return done

In [ ]:
# 7. Same-machine check (20 flights, about 1-3 min): authors' model on 10 hard scenes, compared with the lab GPU.
#    Tells us whether Colab results can be compared directly with the lab results.
import csv
if "released_base" in HAVE:
    evaluate("released_base", "check_hard10_fov82", "check_released_hard10")
    sh(f"python lab_tools/rescore_goal_plane.py {RES / 'check_released_hard10'}", cwd=REPO)
    load = lambda p: {(r["scene_id"], r["speed_mps"]): r for r in csv.DictReader(open(p))}
    colab = load(RES / "check_released_hard10_goal_plane.csv")
    labr = load(RES / "lab_released_base_fov82_goal_plane.csv")
    same = sum(colab[k]["termination"] == labr[k]["termination"] for k in colab)
    print(f"\nSAME-MACHINE CHECK: {same}/{len(colab)} flights ended the same way on Colab and on the lab GPU")

In [ ]:
# 8. Fine-tune (in the background) while evaluating the authors' model, then evaluate the fine-tuned models.
FT_ITERS = 1200      # iterations per fine-tune; the log below shows the speed (it/s)
FT_LR = 3e-4
FINETUNES = [("ft_height", "--ground_affinity_fix --target_dz 1.0"),   # the novelty
             ("ft_fix",    "--ground_affinity_fix")]                   # control: same fine-tune, no height targets
args_file = (SIM / "configs/single_agent.args").read_text().split()

def start_finetune(name, extra):
    log = open(DRIVE / f"{name}_train.log", "a")
    cmd = [sys.executable, "-u", "main_cuda.py", *args_file, "--num_iters", str(FT_ITERS), "--lr", str(FT_LR),
           "--resume", str(CKPT / "released_base.pth"), *extra.split()]
    print(f">>> fine-tuning {name}: {' '.join(cmd[2:])}")
    (SIM / "checkpoint_final.pth").unlink(missing_ok=True)
    return subprocess.Popen(cmd, cwd=SIM, stdout=log, stderr=subprocess.STDOUT), log

def finish_finetune(name, proc, log):
    last = 0
    while proc.poll() is None:
        time.sleep(30)
        text = (DRIVE / f"{name}_train.log").read_text(errors="ignore")
        m = re.findall(r"(\d+)/(\d+) \[([^<\]]+)<([^,\]]+), *([\d.]+ *[a-z/]+)", text)
        if m and time.time() - last > 120:
            i, n, spent, left, rate = m[-1]
            print(f"   {name}: {i}/{n} iterations, {spent} so far, about {left} left ({rate})", flush=True)
            last = time.time()
    log.close()
    if proc.returncode != 0 or not (SIM / "checkpoint_final.pth").exists():
        print((DRIVE / f"{name}_train.log").read_text(errors="ignore")[-3000:])
        raise RuntimeError(f"fine-tuning {name} failed; send the lines above to Claude")
    shutil.copy2(SIM / "checkpoint_final.pth", CKPT / f"{name}.pth")
    shutil.copy2(SIM / "checkpoint_final.pth", CKPT_DRIVE / f"{name}.pth")
    HAVE.add(name)
    print(f">>> {name} saved to Drive ({sha256(CKPT / f'{name}.pth')[:16]}...)")

# while each fine-tune runs, evaluate something that is already available
PARALLEL = {"ft_height": [("released_base", "lab_plan_height_fov82", "released_base_height")],
            "ft_fix":    [("ft_height", "lab_plan_height_fov82", "ft_height_height")]}
for name, extra in FINETUNES:
    if name in HAVE:
        print(f"{name}.pth already on Drive, no fine-tuning needed")
        for job in PARALLEL[name]:
            evaluate(*job)
        continue
    proc, log = start_finetune(name, extra)
    for job in PARALLEL[name]:
        evaluate(*job)
    finish_finetune(name, proc, log)

RUNS = [
    ("released_base", "lab_plan_height_fov82", "released_base_height"),
    ("ft_height",     "lab_plan_height_fov82", "ft_height_height"),
    ("ft_fix",        "lab_plan_height_fov82", "ft_fix_height"),
    ("ft_height",     "lab_plan_fov82",        "ft_height_fov82"),   # standard scenes: safety kept?
    ("ft_fix",        "lab_plan_fov82",        "ft_fix_fov82"),
    ("released_base", "lab_plan_fov82",        "released_base_fov82"),
]
for ckpt, plan, out in RUNS:
    evaluate(ckpt, plan, out)

In [ ]:
# 9. Results: tables + statistics, saved to MyDrive/uav_eval/summary/novelty_results.txt
def complete(name):
    return len(list((RES / name).glob("run_[0-9][0-9][0-9][0-9][0-9]/result.metrics.json"))) == 180

import torch
gpu = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True, capture_output=True, text=True).stdout.strip()
report = [f"Colab run: GPU {gpu}, torch {torch.__version__}, team repo {TEAM_COMMIT}, DiffPhysDrone {SIM_COMMIT[:7]} + PyTorch patch\n"]
def run_and_log(cmd, title):
    r = subprocess.run(cmd, shell=True, cwd=DRIVE, text=True, capture_output=True)
    text = f"\n##### {title}\n$ {cmd}\n{r.stdout}{r.stderr}"
    print(text); report.append(text)

done = [out for _, _, out in RUNS if complete(out)]
print("complete runs:", done)
for out in done:
    subprocess.run(f"python {REPO}/lab_tools/rescore_goal_plane.py benchmark_results/{out}",
                   shell=True, cwd=DRIVE, capture_output=True)
T = f"python {REPO}/lab_tools"
height = [n for n in ["released_base_height", "ft_fix_height", "ft_height_height"] if n in done]
pairs = [(a, b) for a, b in [("ft_fix_height", "ft_height_height"), ("released_base_height", "ft_height_height"),
                             ("released_base_height", "ft_fix_height")] if a in done and b in done]
pair_args = " ".join(f"--pair {a} {b}" for a, b in pairs)
if height:
    run_and_log(f"{T}/height_table.py {' '.join(height)} {pair_args}", "Height-change test (novelty)")
    run_and_log(f"{T}/learning_curve.py {' '.join(height)}", "Height-change test: safe success")
    run_and_log(f"{T}/ten_metrics.py benchmark_results {' '.join(height)} {pair_args} "
                f"--out summary/ten_metrics_height", "Ten metrics, height-change test")
std = [n for n in ["released_base_fov82", "ft_fix_fov82", "ft_height_fov82"] if n in done]
run_and_log(f"{T}/learning_curve.py lab_released_base_fov82 {' '.join(std)}",
            "Standard scenes (lab_* = lab GPU, others = Colab)")
base = "released_base_fov82" if "released_base_fov82" in done else "lab_released_base_fov82"
for a, b in [(base, "ft_height_fov82"), ("ft_fix_fov82", "ft_height_fov82"), (base, "ft_fix_fov82")]:
    if b in done and (a in done or a.startswith("lab_")):
        run_and_log(f"{T}/compare_paired.py {a} {b}", f"Standard scenes, paired: {a} vs {b}")
if std:
    std_pairs = " ".join(f"--pair {a} {b}" for a, b in [("ft_fix_fov82", "ft_height_fov82"),
                         ("released_base_fov82", "ft_fix_fov82")] if a in std and b in std)
    run_and_log(f"{T}/ten_metrics.py benchmark_results {' '.join(std)} {std_pairs} --out summary/ten_metrics_standard",
                "Ten metrics, standard scenes")
(SUMMARY / "novelty_results.txt").write_text("".join(report))
print(f"\nSaved {SUMMARY / 'novelty_results.txt'} (plus ten_metrics_*.md / *.csv). Tell Claude it is done.")